# Challenge — Red neuronal con Keras 3
### Universidad EAFIT | SI3003 — Introducción a la Inteligencia Artificial

---

## Objetivo

En clase construimos una red neuronal para clasificación de imágenes usando **Keras 3**.

En este ejercicio aplicarás **la misma receta** sobre un dataset diferente: **CIFAR-10 convertido a escala de grises**.

No necesitas diseñar un pipeline nuevo. La descarga, conversión a escala de grises y visualización están resueltas. Tu trabajo es completar únicamente las etapas de Keras que vimos en clase:

```text
Datos → Normalización → Modelo → Loss + Optimizer → fit() → evaluate() → Predicción
```

Las celdas marcadas con `# TODO` son las que debes completar.


---
## 0. Librerías


In [ ]:
import keras
from keras import layers
import numpy as np
import matplotlib.pyplot as plt

SEED = 42
keras.utils.set_random_seed(SEED)

print(f"Keras version : {keras.__version__}")
print(f"Backend       : {keras.backend.backend()}")


---
# 1. Dataset: CIFAR-10

CIFAR-10 contiene 50,000 imágenes de entrenamiento y 10,000 de prueba. Las imágenes originales son de **32×32×3** y pertenecen a 10 categorías:

`airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck`.


In [ ]:
# Esta parte está resuelta.
(X_train, y_train), (X_test, y_test) = keras.datasets.cifar10.load_data()

y_train = y_train.squeeze()
y_test = y_test.squeeze()

class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

print("X_train original:", X_train.shape)
print("X_test original :", X_test.shape)
print("y_train         :", y_train.shape)


## 1.1 RGB → escala de grises

Usaremos la transformación:

$$Gray = 0.299R + 0.587G + 0.114B$$

Esta parte está resuelta porque el objetivo del ejercicio es practicar Keras.


In [ ]:
X_train = (
    0.299 * X_train[..., 0] +
    0.587 * X_train[..., 1] +
    0.114 * X_train[..., 2]
)

X_test = (
    0.299 * X_test[..., 0] +
    0.587 * X_test[..., 1] +
    0.114 * X_test[..., 2]
)

print("X_train grayscale:", X_train.shape)
print("X_test grayscale :", X_test.shape)


### Pregunta 1

Después de convertir las imágenes a escala de grises, cada imagen tiene tamaño `32×32`.

¿Cuántos valores tendrá cada imagen después de aplicar `Flatten()`?

**Respuesta:** 32 × 32 = **1024 valores**. `Flatten()` convierte la matriz
2D de 32×32 píxeles en un único vector de 1024 componentes (una por píxel).


## 1.2 Normalización

Los valores de los píxeles están entre 0 y 255. Completa el código para llevarlos al rango `[0,1]`.


In [ ]:
# TODO 1 — Normalización
X_train = X_train.astype("float32") / 255.0
X_test  = X_test.astype("float32") / 255.0

print(X_train.min(), X_train.max())


In [ ]:
# Validación TODO 1
assert X_train.dtype == np.float32
assert X_test.dtype == np.float32
assert X_train.min() >= 0.0
assert X_train.max() <= 1.0
print("✓ Datos normalizados correctamente")


## 1.3 Visualización

Esta parte está resuelta.


In [ ]:
plt.figure(figsize=(12, 6))
for i in range(12):
    plt.subplot(3, 4, i + 1)
    plt.imshow(X_train[i], cmap="gray")
    plt.title(class_names[y_train[i]])
    plt.xticks([])
    plt.yticks([])
plt.tight_layout()
plt.show()


---
# 2. Construir la red neuronal

Construiremos la misma arquitectura conceptual vista en clase:

```text
Imagen 32×32
   ↓
Flatten
   ↓
1024 valores
   ↓
Dense(128)
   ↓
ReLU
   ↓
Dense(10)
   ↓
Logits
```

> La última capa **no** debe tener Softmax. Trabajaremos con logits.


In [ ]:
# TODO 2 — Completa la arquitectura
model = keras.Sequential([
    keras.Input(shape=(32, 32)),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(10)
])

model.summary()


In [ ]:
# Validación TODO 2
expected_params = (1024 * 128 + 128) + (128 * 10 + 10)
assert model.count_params() == expected_params
print(f"✓ Arquitectura correcta: {model.count_params():,} parámetros")


### Preguntas

**2.** ¿Por qué necesitamos `Flatten()` antes de la primera capa `Dense`?

**Respuesta:** Una capa `Dense` espera como entrada un **vector 1D** por
ejemplo. La imagen llega como una matriz 2D de 32×32; `Flatten()` la "aplana"
a un vector de 1024 valores para que cada píxel se conecte a las neuronas de
la capa densa.

**3.** ¿Por qué la última capa tiene 10 neuronas?

**Respuesta:** Porque CIFAR-10 tiene **10 clases**. Cada neurona de salida
produce el *logit* (puntaje) asociado a una clase; la clase predicha es la de
mayor puntaje.

**4.** ¿Qué función cumple `ReLU`?

**Respuesta:** Introduce **no linealidad** ($\mathrm{ReLU}(x)=\max(0,x)$). Sin
una activación no lineal, apilar capas densas equivaldría a una sola
transformación lineal; ReLU permite a la red aprender relaciones complejas y,
además, mitiga el desvanecimiento del gradiente y es barata de calcular.


---
# 3. Configurar el entrenamiento

Usaremos:

- `Adam`
- `SparseCategoricalCrossentropy`
- `accuracy`

Como la red produce logits, la pérdida debe configurarse con `from_logits=True`.


In [ ]:
lr_rate = 0.001

# TODO 3 — Configura el entrenamiento
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=lr_rate),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)


### Pregunta 5

¿Por qué usamos `from_logits=True`?

**Respuesta:** Porque la última capa **no** aplica Softmax: produce *logits*
(valores reales sin normalizar), no probabilidades. Con `from_logits=True` la
función de pérdida aplica internamente el Softmax de forma numéricamente
estable (log-sum-exp) antes de calcular la cross-entropy, evitando problemas
de precisión que surgirían si aplicáramos Softmax manualmente y luego el log.


---
# 4. Entrenar el modelo

Usaremos 10 épocas y mini-batches de 64 ejemplos. Completa `fit()`.


In [ ]:
epochs = 10
batch_size = 64

# TODO 4 — Entrenamiento
history = model.fit(
    X_train,
    y_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(X_test, y_test),
    shuffle=True
)


### Preguntas

**6.** ¿Qué representa una `epoch`?

**Respuesta:** Una **pasada completa** por todo el conjunto de entrenamiento.
En una época el modelo ve (una vez) los 50 000 ejemplos, divididos en
mini-batches, actualizando sus pesos en cada batch.

**7.** ¿Qué representa `batch_size=64`?

**Respuesta:** El número de ejemplos que se procesan juntos antes de hacer
**una** actualización de los pesos. Con 64, el gradiente se promedia sobre 64
imágenes; es un compromiso entre la estabilidad del gradiente (batches grandes)
y la frecuencia de actualización y uso de memoria (batches pequeños).


## 4.1 Curvas de aprendizaje

Esta parte está resuelta.


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history.history["accuracy"], label="Train accuracy")
plt.plot(history.history["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Evolución del entrenamiento")
plt.legend()
plt.show()


### Pregunta 8

Observa las dos curvas. ¿Hay evidencia de overfitting? Justifica brevemente.

**Respuesta:** Sí, hay evidencia de *overfitting* moderado. A lo largo de las
épocas la **accuracy de entrenamiento sigue subiendo** mientras la **accuracy
de validación se estanca** (e incluso baja levemente al final): la brecha
entre ambas curvas crece. Eso indica que el MLP empieza a memorizar el
entrenamiento en lugar de generalizar. Es esperable en una red totalmente
conectada aplicada a imágenes, porque al aplanar se pierde la estructura
espacial y el modelo tiene muchos parámetros para 1024 entradas poco
informativas por sí solas.


---
# 5. Evaluar el modelo

Completa `evaluate()`.


In [ ]:
# TODO 5 — Evaluación
test_loss, test_accuracy = model.evaluate(
    X_test,
    y_test,
    batch_size=batch_size,
    verbose=0
)

print(f"Test loss     : {test_loss:.4f}")
print(f"Test accuracy : {test_accuracy:.4f}")
print(f"Accuracy (%)  : {test_accuracy * 100:.2f}%")


### Pregunta 9

Compara este resultado con Fashion-MNIST, utilizado en clase.

¿CIFAR-10 en escala de grises parece más fácil o más difícil para esta red? Justifica usando el accuracy obtenido.

**Respuesta:** **Más difícil.** Con el mismo MLP, Fashion-MNIST suele alcanzar
~88–90 % de accuracy, mientras que CIFAR-10 en escala de grises con esta red
densa se queda alrededor de ~36–40 %. CIFAR-10 son fotografías naturales con
fondos variados, distintas poses, iluminación y texturas; además, al pasar a
escala de grises perdemos el color, que es una pista fuerte (p. ej. para cielo,
césped). Fashion-MNIST son objetos centrados sobre fondo negro y mucho más
homogéneos, por lo que es un problema considerablemente más sencillo para una
red totalmente conectada.


---
# 6. Logits y Softmax

La salida de la red son 10 logits. Para interpretarlos como probabilidades necesitamos Softmax.


In [ ]:
index = 0
image = X_test[index]
true_class = y_test[index]

plt.imshow(image, cmap="gray")
plt.title(f"Clase real: {class_names[true_class]}")
plt.axis("off")
plt.show()

image_batch = np.expand_dims(image, axis=0)
logits = model(image_batch, training=False)

print("Logits:")
print(keras.ops.convert_to_numpy(logits))


In [ ]:
# TODO 6 — Logits -> probabilidades
probabilities = keras.ops.softmax(logits)

# TODO 7 — Clase con mayor probabilidad
prediction = keras.ops.argmax(probabilities, axis=1)
prediction = int(keras.ops.convert_to_numpy(prediction)[0])

print("Clase real     :", class_names[true_class])
print("Clase predicha :", class_names[prediction])


In [ ]:
probabilities_np = keras.ops.convert_to_numpy(probabilities)[0]

print("\nProbabilidades:")
for i, p in enumerate(probabilities_np):
    marker = " ← predicción" if i == prediction else ""
    print(f"{class_names[i]:12s}: {p:.4f}{marker}")

print("\nSuma:", probabilities_np.sum())


### Pregunta 10

¿Por qué las probabilidades producidas por `Softmax` deben sumar aproximadamente 1?

**Respuesta:** Porque Softmax normaliza los logits a una **distribución de
probabilidad** sobre las clases: $\mathrm{softmax}(z)_i = e^{z_i}/\sum_j e^{z_j}$.
Al dividir cada exponencial por la suma de todas, los valores quedan en $[0,1]$
y, por construcción, **suman exactamente 1** (salvo mínimos errores de
redondeo). Representan la confianza del modelo repartida entre las 10 clases
mutuamente excluyentes.


---
# 7. Varias predicciones

Esta parte está resuelta. Observa especialmente los errores del modelo.


In [ ]:
n = 12
logits_batch = model(X_test[:n], training=False)
predictions = keras.ops.argmax(logits_batch, axis=1)
predictions = keras.ops.convert_to_numpy(predictions)

plt.figure(figsize=(12, 8))
for i in range(n):
    plt.subplot(3, 4, i + 1)
    plt.imshow(X_test[i], cmap="gray")
    real = class_names[y_test[i]]
    pred = class_names[predictions[i]]
    plt.title(f"Real: {real}\nPred: {pred}", fontsize=9)
    plt.xticks([])
    plt.yticks([])
plt.tight_layout()
plt.show()


---
# 8. Reflexión final

### 1. ¿Cuál fue el accuracy final?

**Respuesta:** El valor exacto es el `test_accuracy` impreso en la sección 5
(con este MLP y 10 épocas en escala de grises, típicamente **~36–40 %**).

### 2. ¿Qué clases parecen más difíciles de distinguir?

**Respuesta:** Las clases de **animales con siluetas y texturas parecidas en
gris** (p. ej. `cat`/`dog`/`deer`/`bird`) y los vehículos entre sí
(`automobile`/`truck`). Al perder el color y la estructura espacial, formas
similares se confunden; esto se observa en la cuadrícula de predicciones y
sería más claro en una matriz de confusión.

### 3. Después de `Flatten()`, la imagen se convierte en un vector de 1024 valores. ¿Qué información importante sobre la imagen podría perderse al tratarla únicamente como un vector?

**Respuesta:** Se pierde la **estructura espacial 2D**: la relación de
vecindad entre píxeles (qué píxel está al lado/arriba/abajo de otro). Bordes,
formas y patrones locales dejan de ser explícitos; para la red densa el píxel
(0,0) y el (31,31) son solo dos posiciones del vector sin noción de distancia
ni de patrones locales que se repiten en distintas partes de la imagen.

### 4. ¿Qué tipo de arquitectura podría aprovechar mejor la estructura espacial de las imágenes?

**Respuesta:** Las **Redes Neuronales Convolucionales (CNNs)**. Sus filtros
convolucionales operan sobre vecindades locales y comparten pesos a lo largo de
la imagen, de modo que detectan bordes, texturas y formas independientemente de
su posición (invarianza a traslación) y con muchos menos parámetros que un MLP.
